> Historical CUDA/Colab implementation. The published thesis results use the local MPS implementation in `src/`, not this historical training backend. Run in a fresh runtime.

# HateWiC Formal Experiments
Protocol: 26 October 2026. This independent notebook implements the approved v7.2 method, two ten-fold protocols, seven neural configurations and linked-gloss evaluation.

The 24 October foundation notebook remains unchanged. Run sections in order. This notebook reads private research files and writes only into `HateWiC_Project/formal_20261026`. Do not make its data or saved outputs public.

The first pass has 140 neural fits; four additional core seeds add 320. Free GPU availability and the measured pilot determine feasible progress. No paid service is used. A saved run is resumed rather than silently replaced.

## 1. Prepare a clean environment
Select **Runtime → Change runtime type → T4 GPU**. Install dependencies before importing them. The installed PyTorch version and device are recorded; no service upgrade is required.

In [ ]:
import os, sys, subprocess
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
    'numpy==2.5.4','pandas==3.1.0','scipy==1.18.1','scikit-learn==1.9.1',
    'transformers==5.20.2','pytest==9.1.1','nbformat==5.11.1','torch>=2.6,<3'])
print('Dependencies ready. If Colab requests a runtime restart, restart and continue from Section 2.')

## 2. Mount Drive and load the reviewed historical source
The original embedded archive is expanded under `historical_colab/` in this repository. Its SHA-256 and file hashes are recorded in `docs/historical_archive_manifest.json`. Configure the private project path before continuing.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
REPO=Path('/content/HateWiC-Masters-Research')
if not REPO.exists():
    subprocess.check_call(['git','clone','https://github.com/ZSJJessey/HateWiC-Masters-Research.git',str(REPO)])
PROJECT=Path(os.environ.get('HATEWIC_PROJECT','/content/drive/MyDrive/HateWiC_Project'))
FORMAL=Path(os.environ.get('HATEWIC_FORMAL_OUTPUT',str(PROJECT/'formal_public_reproduction')))
FORMAL.mkdir(parents=True,exist_ok=True)
WORK=REPO/'historical_colab'
sys.path.insert(0,str(WORK/'src'))
for module_name in list(sys.modules):
    if module_name=='hatewic_formal' or module_name.startswith('hatewic_formal.'):
        del sys.modules[module_name]
from hatewic_formal.config import write_json,sha256,code_hash
from hatewic_formal.runner import environment,progress
CACHE=Path('/content/hatewic_hf_cache');CACHE.mkdir(parents=True,exist_ok=True)
write_json(FORMAL/'environment.json',environment())
print(environment())
print('Private reproduction output directory:',FORMAL)


## 3. Verify historical evidence and rebuild the conservative cohort
Read the completed review files without reopening the review interface. Preserve all original rows. Exclude unresolved positions, rule-selected occurrences and non-three-annotator examples from this version.

In [ ]:
from hatewic_formal.data import rebuild,verify_position_sources
from hatewic_formal.folds import create_folds
RAW=PROJECT/'data/raw/HateWiC_IndividualAnnos.csv'
REVIEW=PROJECT/'data/processed/review_03_20261019T022652709994Z'
LEDGER=PROJECT/'results/preparation_07_20261021T114013264380Z/position_status_ledger.csv'
PAIRS=PROJECT/'results/duplicate_audit_09_20261021T114331309559Z/near_duplicate_candidate_pairs.csv'
evidence=verify_position_sources(RAW,REVIEW/'target_span_candidates_v03.csv',
    REVIEW/'target_review_reduced_05b/target_review_records.json',
    REVIEW/'target_review_reduced_05b/rule_assigned_repeated_candidates.csv',LEDGER)
write_json(FORMAL/'artifacts/position_evidence_audit.json',evidence)
main,scoring,audit=rebuild(RAW,LEDGER,PAIRS,FORMAL/'artifacts')
assert sha256(FORMAL/'artifacts/main.csv')=='ae159dcbfd94c3803045696c704b981f08f5e44fd2c6b198f3f895ecb43435ca', 'Cloud/local main-set parity failed; stop before scoring.'
manifest=create_folds(main,FORMAL/'artifacts')
import pandas as pd
main=pd.read_csv(FORMAL/'artifacts/main.csv',keep_default_na=False)
print('Main cohort verified:',len(main))
print('Class support:',audit['main_label_counts'])
print('Both ten-fold protocols passed. Scoring metadata are stored separately.')

## 4. Run implementation tests and full tokenizer checks
Synthetic tests cover pooling, objectives, gradients, freezing, exact recovery, lexical-prior ties, metrics and clustered inference. Full-data checks validate both target-preserving token windows. No formal model is fitted in this section.

In [ ]:
import subprocess, os
test_env={**os.environ,'PYTHONPATH':str(WORK/'src'),'CUDA_VISIBLE_DEVICES':''}
tested=subprocess.run([sys.executable,'-m','pytest',str(WORK/'tests'),'-q'],env=test_env,text=True,capture_output=True)
print(tested.stdout)
if tested.returncode:
    print(tested.stderr)
    raise RuntimeError('Implementation checks failed.')
write_json(FORMAL/'verification/tests.json',{'status':'passed','code_sha256':code_hash(),'output':tested.stdout,'scope':'synthetic CPU implementation tests; GPU checked by resource pilot'})
from hatewic_formal.tokenization import create_tokenizers
from hatewic_formal.feature_extraction import audit_tokenizers,extract_frozen_features,load_features
tokenizers=create_tokenizers(cache_dir=str(CACHE))
token_audit=audit_tokenizers(main,tokenizers,FORMAL/'artifacts')
token_audit.update(data_version=audit['data_version'],main_examples=len(main))
write_json(FORMAL/'artifacts/tokenizer_audit.json',token_audit)
print('Full tokenizer audit passed:',token_audit['main_examples'])
print('Context clipping:',token_audit['truncated_contexts'],'Gloss clipping:',token_audit['gloss_truncated'])

## 5. Run the free-GPU resource pilot
Benchmark 64 training and 32 validation examples for the contextual model and full model. No test examples enter the pilot. Save actual time, memory, checkpoint size and available storage before scheduling formal neural runs.

In [ ]:
from hatewic_formal.runner import resource_pilot,check_gates,make_schedule
pilot=resource_pilot(FORMAL,tokenizers,cache_dir=str(CACHE))
check_gates(FORMAL)
schedule=make_schedule(FORMAL)
print('Launch gates passed.')
for item in pilot['benchmarks']:
    print(item['model'],'pilot seconds:',round(item['elapsed_seconds'],1),
          'projected max 8-epoch fold hours:',round(item['projected_max_8epoch_seconds_per_full_fold']/3600,2))
print(progress(FORMAL))

## 6. Run classical baselines on the fixed folds
Three baselines require CPU only. Each estimator is fitted on its current training partition. Test predictions are saved once per rotation and verified against the frozen manifest.

In [ ]:
from hatewic_formal.baselines import run_baselines
from hatewic_formal.evaluation import summarize_oof
fold_hash=sha256(FORMAL/'artifacts/fold_manifest.csv')
classical=run_baselines(main,manifest,FORMAL/'baselines/classical',audit['data_version'],fold_hash)
classical_summary=summarize_oof(classical,scoring)
classical_summary.to_csv(FORMAL/'baselines/classical/metrics.csv',index=False)
display(classical_summary[['protocol','model','subgroup','macro_f1','n']])

## 7. Extract frozen representations and run the two feature MLPs
Each pinned backbone is extracted once. Cached features are reused only after ID, input, code and model checks. RO1 uses equal-weight eligible target/POS groups. MLP scaling is fitted separately within each training partition.

In [ ]:
feature_paths=extract_frozen_features(main,tokenizers,str(CACHE),FORMAL/'frozen_features')
from hatewic_formal.reporting import geometry_report
display(geometry_report(FORMAL,feature_paths))
features={'bert':load_features(feature_paths['bert_base']),'hatebert':load_features(feature_paths['hatebert'])}
mlp=run_baselines(main,manifest,FORMAL/'baselines/frozen_mlp',audit['data_version'],fold_hash,
    models=['bert_frozen_mlp','hatebert_frozen_mlp'],features_by_backbone=features)
summarize_oof(mlp,scoring).to_csv(FORMAL/'baselines/frozen_mlp/metrics.csv',index=False)
print('Frozen-feature baseline predictions verified.')

## 8. Run the single-seed neural schedule
The queue contains seven configurations × two protocols × ten rotations. It resumes the active job from its last saved optimizer boundary. After each job, test predictions are verified and temporary checkpoints are cleaned. Leave this cell running while the free GPU is available; rerun it after reconnecting. It never purchases compute or omits queued comparisons.

In [ ]:
from hatewic_formal.runner import run_pending
status=run_pending(FORMAL,tokenizers,phase='single_seed',max_jobs=None,cache_dir=str(CACHE))
print(status)

## 9. Add core seeds after the full single-seed pass
Complete all 140 first-pass jobs before starting the additional 320 core jobs. Resource reports show the remaining count and measured GPU-hour projection. An unavailable GPU is reported as a capacity gap.

In [ ]:
schedule=make_schedule(FORMAL)
missing=[rid for rid in schedule.loc[schedule.phase.eq('single_seed'),'run_id']
         if not (FORMAL/'runs'/rid/'prediction_verified.json').is_file()]
if missing: raise RuntimeError(f'{len(missing)} single-seed runs remain; finish Section 8 first.')
print(run_pending(FORMAL,tokenizers,phase='core_repeats',max_jobs=None,cache_dir=str(CACHE)))

## 10. Generate matched summaries and the ten-test Holm family
Single-seed and five-seed reports remain separate. Missing or unsupported comparisons do not become significance claims. Confidence intervals condition on fitted models; overlapping training folds are not independent repeats.

In [ ]:
from hatewic_formal.reporting import final_reports
display(final_reports(FORMAL,phase='single_seed'))
print('Five-seed report is generated only when every core model/seed/protocol has complete OOF coverage.')
if all((FORMAL/'runs'/rid/'prediction_verified.json').is_file() for rid in schedule.run_id):
    display(final_reports(FORMAL,phase='core_repeats'))

## 11. Save the progress inventory for Chapter 4
Read `run_progress.json`, the result CSVs and their hashes. Insert measured results into the separate Chapter 4 draft. Preserve `Pending experimental results` for analyses that have not run. Experiment target: 15 November; Chapter 4 draft: 20 November.

In [ ]:
print(progress(FORMAL))
print('Restricted result directory:',FORMAL)
print('Download this notebook via File → Download → Download .ipynb after saving outputs.')